## Import and Installs

In [0]:
import sys
import os
from pyspark.sql import functions as F

sys.path.append(os.path.abspath(".."))

from src.config.settings import LakehouseConfig
from src.gold.aggregations import build_gold_metricas_cooperados, build_gold_desempenho_canais

In [0]:
config = LakehouseConfig()
spark.sql(f"USE CATALOG {config.catalog}")
spark.sql(f"USE SCHEMA {config.schema_gold}")

## read tabelas silver

In [0]:
df_dim_cliente = spark.table(
    f"{config.catalog}.{config.schema_silver}.dim_cliente_scd2"
)
df_transacoes = spark.table(
    f"{config.catalog}.{config.schema_silver}.silver_transacoes"
)
df_cartoes = spark.table(f"{config.catalog}.{config.schema_bronze}.bronze_cartoes_cdc")
df_contas = spark.table(f"{config.catalog}.{config.schema_bronze}.bronze_contas_cdc")

## métricas 

In [0]:
gold_cooperados = build_gold_metricas_cooperados(
    df_dim_cliente=df_dim_cliente,
    df_transacoes=df_transacoes,
    df_cartoes=df_cartoes,
    df_contas=df_contas
)

(
    gold_cooperados.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{config.catalog}.{config.schema_gold}.gold_metricas_cooperados")
)

print("Tabela gold_metricas_cooperados gravada com sucesso!")

## desempenho por canais

In [0]:
gold_canais = build_gold_desempenho_canais(df_transacoes)

(
    gold_canais.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{config.catalog}.{config.schema_gold}.gold_desempenho_canais")
)

print("Tabela gold_desempenho_canais gravada com sucesso")